# Lipsyncstudio on Kaggle (LatentSync 1.5, 16 GB GPU)

Runs the full `local_ui.py` dubbing studio and exposes it as a public `*.gradio.live` link.

This notebook builds a **Python 3.10** environment with [`uv`](https://github.com/astral-sh/uv) and installs the project's *exact, proven* dependency lock into it, because the pinned versions (scikit-image, decord, numba, ...) only ship wheels for 3.10 and would try to build from source on Kaggle's newer Python.

## Before you Run All
1. **Settings -> Accelerator:** `GPU P100` or `GPU T4 x2` (16 GB). *Disabled? Phone-verify at kaggle.com/settings.*
2. **Settings -> Internet:** `On` (same phone-verification gate). Required for clone, uv, pip, and model downloads.
3. **Add-ons -> Secrets:** add a secret named **`SIXTYDB_API_KEY`** with your 60db key and attach it.
4. **Run All.** First run takes ~10-15 min (env build + ~5 GB checkpoints). The last cell blocks and prints `Running on public URL: https://xxxx.gradio.live` - open it.

The share link is public to anyone who has it and spends your 60db credits. It lives only while this session runs.

## 1. Clone the app and system libs

In [ ]:
import os, subprocess, pathlib

REPO = "https://github.com/RajEmaster/60db-lipsync-studio.git"
CLONE = "/kaggle/working/60db-lipsync-studio"
APP = CLONE + "/LatentSync-trial"

if not os.path.isdir(CLONE):
    subprocess.run(["git", "clone", "--depth", "1", REPO, CLONE], check=True)
else:
    subprocess.run(["git", "-C", CLONE, "pull", "--ff-only"], check=False)

subprocess.run("apt-get -qq update && apt-get -qq install -y libgl1 ffmpeg",
               shell=True, check=False)

os.chdir(APP)
print("App dir:", APP)

## 2. Build a Python 3.10 environment with uv
`uv` downloads a standalone CPython 3.10 and creates an isolated venv at `/kaggle/working/env310`.

In [ ]:
ENV = "/kaggle/working/env310"
PY = ENV + "/bin/python"

subprocess.run(["pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "venv", "--python", "3.10", ENV], check=True)
print(subprocess.run([PY, "--version"], capture_output=True, text=True).stdout.strip())

## 3. Install the proven stack into the 3.10 env
torch 2.5.1+cu121 first, then the exact project lock (read from the repo) + the UI deps.

In [ ]:
# 3a. torch / torchvision / torchaudio from the cu121 index
subprocess.run([
    "uv", "pip", "install", "--python", PY,
    "torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
    "--index-url", "https://download.pytorch.org/whl/cu121",
], check=True)

In [ ]:
# 3b. Build requirements_kaggle.txt from the repo's own proven lock + UI reqs
lock = pathlib.Path("trial-requirements-lock.txt").read_text(encoding="utf-16")
ui = pathlib.Path("requirements.txt").read_text(encoding="utf-8")

lines, have = [], set()
def name_of(s):
    return s.split("==")[0].split(">")[0].split("<")[0].strip().lower().replace("_", "-")

# Pin ONLY model-critical packages; keep everything else present but UNPINNED so
# the resolver can satisfy gradio 5.49.1 (it needs pillow<12 AND
# huggingface-hub>=0.33.5, both of which the worker lock pinned incompatibly).
KEEP = {
    "numpy", "diffusers", "transformers", "tokenizers", "accelerate", "omegaconf",
    "einops", "decord", "kornia", "kornia-rs", "opencv-python", "onnx", "onnxruntime",
    "scikit-image", "scikit-learn", "scipy", "librosa", "numba", "llvmlite", "soundfile",
    "safetensors", "imageio", "imageio-ffmpeg", "ffmpeg-python",
    "tiktoken", "gradio", "realtimevad", "kaldi-native-fbank", "sixtydb",
    "python-dotenv", "psutil",
}

def emit(p):
    n = name_of(p)
    if n in have:
        return
    have.add(n)
    lines.append(p if n in KEEP else n)  # keep version only for critical pkgs

for ln in lock.splitlines():
    p = ln.strip()
    if not p or p.startswith("#"):
        continue
    if p.lower().startswith(("torch==", "torchvision==", "torchaudio==")):
        continue  # installed from the cu121 index above
    emit(p)

for ln in ui.splitlines():
    p = ln.strip()
    if p and not p.startswith("#"):
        emit(p)

pathlib.Path("requirements_kaggle.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

In [ ]:
# 3c. Install the locked stack (~5-8 min)
subprocess.run(["uv", "pip", "install", "--python", PY,
                "-r", "requirements_kaggle.txt"], check=True)

In [ ]:
# 3d. Demucs for 'Restore original background' (optional). Pin numpy so demucs
# can't bump it back to 2.x. If this fails, the app still works - just leave that
# box unchecked in the UI.
subprocess.run(["uv", "pip", "install", "--python", PY,
                "demucs==4.0.1", "numpy==1.26.4"], check=False)

## 4. Patch for Linux / this environment
The app spawns its GPU worker and the Demucs separator via `settings.WORKER_PYTHON`,
a hardcoded Windows `.venv` path. Repoint it at the 3.10 env's Python.

In [ ]:
import re
s = pathlib.Path("settings.py")
txt = s.read_text()
# Replace whatever the WORKER_PYTHON line is now (original Windows path OR an
# already-patched line from a previous run) -> idempotent, re-run safe.
new, n = re.subn(r"(?m)^WORKER_PYTHON\s*=.*$",
                 f"WORKER_PYTHON = Path(r'{PY}')", txt)
assert n >= 1, "No WORKER_PYTHON assignment found in settings.py"
s.write_text(new)
print(s.read_text())

## 5. 60db credentials from the Kaggle secret

In [ ]:
from kaggle_secrets import UserSecretsClient

try:
    key = UserSecretsClient().get_secret("SIXTYDB_API_KEY").strip()
except Exception as exc:
    raise SystemExit(
        "Add a Kaggle secret named SIXTYDB_API_KEY (Add-ons -> Secrets) and attach it "
        "to this notebook, then Run All again. Original error: " + str(exc))

pathlib.Path(".env").write_text(
    f"SIXTYDB_API_KEY={key}\n"
    "SIXTYDB_SDK_BASE_URL=https://api.60db.ai\n"
    "SIXTYDB_TRANSLATION_MODEL=60db-tiny\n")
print("Wrote .env with SIXTYDB_API_KEY (len=%d)" % len(key))

## 6. Download LatentSync 1.5 checkpoints + face models (~5 GB)

In [ ]:
import sys
os.chdir(APP)
# Pure-HTTP downloaders: run with the base-kernel Python (always has 'requests'),
# independent of the env310 install state.
subprocess.run([sys.executable, "download_trial.py"], check=True)
subprocess.run([sys.executable, "download_faces.py"], check=True)
subprocess.run("ls -lh checkpoints checkpoints/vae checkpoints/whisper "
               "checkpoints/auxiliary/models/buffalo_l", shell=True)

In [ ]:
# GPU sanity check inside the 3.10 env
print(subprocess.run(
    [PY, "-c", "import torch;print('CUDA', torch.cuda.is_available(),"
     "torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"],
    capture_output=True, text=True).stdout)

launcher = """import os
os.environ["MPLBACKEND"] = "Agg"  # override the Jupyter 'inline' backend the subprocess inherits
APP = os.path.dirname(os.path.abspath(__file__))
os.chdir(APP)
from local_ui import build_ui
build_ui().queue(default_concurrency_limit=1).launch(
    server_name="0.0.0.0", server_port=7860, share=True,
    allowed_paths=[os.path.join(APP, "trial_inputs"), os.path.join(APP, "local_jobs")],
)
"""
pathlib.Path(APP, "run_kaggle.py").write_text(launcher)
print("Launching... watch for the https://....gradio.live URL below.")

In [ ]:
launcher = """import os
APP = os.path.dirname(os.path.abspath(__file__))
os.chdir(APP)
from local_ui import build_ui
build_ui().queue(default_concurrency_limit=1).launch(
    server_name="0.0.0.0", server_port=7860, share=True,
    allowed_paths=[os.path.join(APP, "trial_inputs"), os.path.join(APP, "local_jobs")],
)
"""
pathlib.Path(APP, "run_kaggle.py").write_text(launcher)
print("Launching... watch for the https://....gradio.live URL below.")

In [ ]:
!cd {APP} && {PY} run_kaggle.py